# 03 Normalize Vital Signs

This notebook transforms raw `observations.csv` into a normalized `vital_signs.csv` dataset. Each measurement becomes one row and is linked to `patient.id` through the raw source patient id.

In [ ]:
import pandas as pd
import numpy as np
import uuid
from pathlib import Path

RAW_DATA_DIR = Path('data/raw')
PROCESSED_DATA_DIR = Path('data/processed')

OBSERVATIONS_FILE = RAW_DATA_DIR / 'observations.csv'
PATIENT_FILE = PROCESSED_DATA_DIR / 'patient.csv'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
VITAL_SIGN_CODE_MAP = {
    '8480-6': 'BLOOD_PRESSURE_SYSTOLIC',
    '8462-4': 'BLOOD_PRESSURE_DIASTOLIC',
    '8867-4': 'HEART_RATE',
    '8310-5': 'TEMPERATURE',
    '8302-2': 'HEIGHT',
    '29463-7': 'WEIGHT',
    '39156-5': 'BMI',
    '2339-0': 'GLUCOSE',
    '2093-3': 'CHOLESTEROL',
    '2708-6': 'OXYGEN_SATURATION',
}


def normalize_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace(r'^\s*$', np.nan, regex=True)


def normalize_unit(value: str) -> str:
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    unit_map = {
        'kg': 'kg',
        'g': 'g',
        'cm': 'cm',
        'm': 'm',
        'Cel': 'C',
        'C': 'C',
        'mm[Hg]': 'mmHg',
        '/min': 'beats/min',
        '%': '%',
        'kg/m2': 'kg/m2',
        'mg/dL': 'mg/dL',
        'mmol/L': 'mmol/L',
    }
    return unit_map.get(value, value)

In [ ]:
observations_df = pd.read_csv(OBSERVATIONS_FILE)
patient_df = pd.read_csv(PATIENT_FILE)

print('Raw observations shape:', observations_df.shape)
print('Patient table shape:', patient_df.shape)
observations_df.head()

In [ ]:
rename_map = {
    'DATE': 'measured_at',
    'PATIENT': 'source_patient_id',
    'ENCOUNTER': 'source_encounter_id',
    'CODE': 'observation_code',
    'DESCRIPTION': 'description',
    'VALUE': 'value',
    'UNITS': 'unit',
    'TYPE': 'source_type',
}

vitals_df = observations_df.rename(columns=rename_map).copy()
vitals_df = normalize_empty_strings(vitals_df)

required_columns = [
    'measured_at',
    'source_patient_id',
    'source_encounter_id',
    'observation_code',
    'description',
    'value',
    'unit',
    'source_type',
]

existing_columns = [col for col in required_columns if col in vitals_df.columns]
vitals_df = vitals_df[existing_columns].copy()
vitals_df.head()

In [ ]:
vitals_df = vitals_df[vitals_df['observation_code'].astype(str).isin(VITAL_SIGN_CODE_MAP.keys())].copy()

vitals_df['vital_type'] = vitals_df['observation_code'].astype(str).map(VITAL_SIGN_CODE_MAP)
vitals_df['measured_at'] = pd.to_datetime(vitals_df['measured_at'], errors='coerce')
vitals_df['value'] = pd.to_numeric(vitals_df['value'], errors='coerce')
vitals_df['unit'] = vitals_df['unit'].apply(normalize_unit)

vitals_df = vitals_df.dropna(subset=['source_patient_id', 'vital_type', 'measured_at', 'value']).copy()

print('Filtered vital signs shape:', vitals_df.shape)
vitals_df.head()

In [ ]:
vital_signs_df = vitals_df.merge(
    patient_df[['id', 'source_patient_id']],
    on='source_patient_id',
    how='inner'
)

vital_signs_df = vital_signs_df.rename(columns={'id': 'patient_id'})
vital_signs_df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(vital_signs_df))])

vital_signs_df = vital_signs_df[[
    'id',
    'patient_id',
    'vital_type',
    'value',
    'unit',
    'measured_at',
    'source_patient_id',
    'source_encounter_id',
    'observation_code',
    'description',
]].copy()

vital_signs_df = vital_signs_df.sort_values(by=['patient_id', 'measured_at', 'vital_type']).reset_index(drop=True)
vital_signs_df.head()

In [ ]:
print('Vital signs null counts:')
print(vital_signs_df.isna().sum())

print('\nVital sign types:')
print(vital_signs_df['vital_type'].value_counts())

print('\nRows without patient_id:', vital_signs_df['patient_id'].isna().sum())
print('Duplicate rows:', vital_signs_df.duplicated(subset=['patient_id', 'vital_type', 'value', 'measured_at']).sum())

In [ ]:
vital_signs_output_file = PROCESSED_DATA_DIR / 'vital_signs.csv'
vital_signs_df.to_csv(vital_signs_output_file, index=False)

print('Exported:', vital_signs_output_file)

In [ ]:
display(vital_signs_df.head(20))